Joining static datas into one

In [1]:
import geopandas as gpd

In [2]:
wind_farm_elevation =  gpd.read_parquet(
    "../data/interim/terrain_processed/wind_farm_elevation.parquet"
)

In [3]:
wind_farm_elevation.count()

Site Name                      825
Technology Type                825
Installed Capacity (MWelec)    825
Turbine Capacity (MW)          812
No. of Turbines                825
Height of Turbines (m)         110
Development Status             825
Region                         825
Country                        825
X-coordinate                   825
Y-coordinate                   825
geometry                       825
Elevation_m                    825
dtype: int64

In [4]:
wind_cornie = gpd.read_parquet(
    "../data/interim/cornie_processed/wind_cornie.parquet"
)

In [5]:
wind_cornie.count()

Site Name                      825
Technology Type                825
Installed Capacity (MWelec)    825
Turbine Capacity (MW)          812
No. of Turbines                825
Height of Turbines (m)         110
Development Status             825
Region                         825
Country                        825
X-coordinate                   825
Y-coordinate                   825
geometry                       825
index_right                    776
Code_18                        776
landcover_status               825
dtype: int64

In [6]:
wind_farms_spatial = (
    wind_farm_elevation
    .merge(
        wind_cornie[
            [
                "Site Name",
                "Installed Capacity (MWelec)",
                "Code_18",
                "landcover_status"
            ]
        ],
        on=[
            "Site Name",
            "Installed Capacity (MWelec)"
        ],
        how="left",
        validate="one_to_one"
    )
)

In [7]:
wind_farms_spatial.count()

Site Name                      825
Technology Type                825
Installed Capacity (MWelec)    825
Turbine Capacity (MW)          812
No. of Turbines                825
Height of Turbines (m)         110
Development Status             825
Region                         825
Country                        825
X-coordinate                   825
Y-coordinate                   825
geometry                       825
Elevation_m                    825
Code_18                        776
landcover_status               825
dtype: int64

In [8]:
wind_farms_spatial.to_parquet(
    "../data/interim/spatial_processed/wind_farms_spatial.parquet",
    index=False
)

In [12]:
import geopandas as gpd

In [9]:
wind_farms_spatial = gpd.read_parquet(
    "../data/interim/spatial_processed/wind_farms_spatial.parquet"
)

In [10]:
wind_farms_spatial.head()

,Site Name,Technology Type,Installed Capacity (MWelec),Turbine Capacity (MW),No. of Turbines,Height of Turbines (m),Development Status,Region,Country,X-coordinate,Y-coordinate,geometry,Elevation_m,Code_18,landcover_status
0,Hywind Scotland Pilot Park (Hywind 2) Demonstr...,Wind Offshore,30,6,5,None,Operational,Offshore,Scotland,433500,846500,POINT (-1.44258 57.50746),0.0,None,Matched
1,Beatrice Demonstrator,Wind Offshore,10,5,2,None,Operational,Offshore,Scotland,347955,929690,POINT (-2.88843 58.25281),0.0,None,Matched
2,Burbo Bank,Wind Offshore,90,3.6,25,None,Operational,Offshore,England,321476,399706,POINT (-3.18492 53.48818),0.0,None,Matched
3,Gunfleet Sands - (Demo) Extension,Wind Offshore,12,6,2,None,Operational,Offshore,England,620650,205297,POINT (1.19191 51.70301),0.0,None,Matched
4,Gunfleet Sands II,Wind Offshore,64.8,3.6,18,None,Operational,Offshore,England,624262,208150,POINT (1.24592 51.72719),0.0,None,Matched


In [11]:
import xarray as xr

weather_feature = xr.open_dataset(
    "../data/interim/era5_processed/weather_feature.nc"
)

weather_feature.head()

<xarray.Dataset> Size: 6kB
Dimensions:            (valid_time: 5, latitude: 5, longitude: 5)
Coordinates:
    number             int64 8B ...
  * valid_time         (valid_time) datetime64[ns] 40B 2018-01-01 ... 2018-01...
  * latitude           (latitude) float64 40B 61.0 60.75 60.5 60.25 60.0
  * longitude          (longitude) float64 40B -9.0 -8.75 -8.5 -8.25 -8.0
    expver             (valid_time) <U4 80B ...
Data variables:
    ws100              (valid_time, latitude, longitude) float32 500B ...
    wd100              (valid_time, latitude, longitude) float32 500B ...
    ws10               (valid_time, latitude, longitude) float32 500B ...
    wd10               (valid_time, latitude, longitude) float32 500B ...
    wind_shear         (valid_time, latitude, longitude) float32 500B ...
    relative_humidity  (valid_time, latitude, longitude) float32 500B ...
    air_density        (valid_time, latitude, longitude) float32 500B ...
    blh                (valid_time, latitude, longitude) float32 500B ...
    tcc                (valid_time, latitude, longitude) float32 500B ...
    tp                 (valid_time, latitude, longitude) float32 500B ...
    ssrd               (valid_time, latitude, longitude) float32 500B ...
    i10fg              (valid_time, latitude, longitude) float32 500B ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-07-31T17:43 GRIB to CDM+CF via cfgrib-0.9.1...

In [12]:
import pandas as pd

In [13]:
farm_grid_mapping = []

for _, row in wind_farms_spatial.iterrows():

    point = row.geometry

    lon = point.x
    lat = point.y

    nearest = weather_feature.sel(
        latitude=lat,
        longitude=lon,
        method="nearest"
    )

    farm_grid_mapping.append({
        "Site Name": row["Site Name"],
        "Installed Capacity (MWelec)":
            row["Installed Capacity (MWelec)"],

        "farm_lat": lat,
        "farm_lon": lon,

        "era5_lat": float(nearest.latitude),

        "era5_lon": float(nearest.longitude)
    })

In [14]:
farm_grid_mapping = pd.DataFrame(
    farm_grid_mapping
)

In [15]:
farm_grid_mapping.head()

,Site Name,Installed Capacity (MWelec),farm_lat,farm_lon,era5_lat,era5_lon
0,Hywind Scotland Pilot Park (Hywind 2) Demonstr...,30,57.507463,-1.442575,57.50,-1.50
1,Beatrice Demonstrator,10,58.252812,-2.888431,58.25,-3.00
2,Burbo Bank,90,53.488183,-3.184923,53.50,-3.25
3,Gunfleet Sands - (Demo) Extension,12,51.703009,1.191906,51.75,1.25
4,Gunfleet Sands II,64.8,51.727189,1.245925,51.75,1.25


In [39]:
farm_grid_mapping.to_parquet(
    "../data/interim/weather_processed/farm_grid_mapping.parquet",
    index=False
)

In [16]:
point = wind_farms_spatial.geometry.iloc[0]

lon = point.x
lat = point.y

In [17]:
farm_grid_mapping[
    ["era5_lat", "era5_lon"]
].drop_duplicates().shape

(305, 2)

In [18]:
farm_grid_mapping[
    "Installed Capacity (MWelec)"
].describe()

count     825
unique    272
top       1.5
freq       29
Name: Installed Capacity (MWelec), dtype: object

In [19]:
farm_grid_mapping[
    farm_grid_mapping["era5_lat"] == 50.25
][
    ["Site Name", "Installed Capacity (MWelec)"]
]

,Site Name,Installed Capacity (MWelec)
142,Carn Vean,10
201,Roskrow Wind Turbines,1.7
299,Four Burrows Wind Farm,4.5
432,Garlenick Estate 2 - Wind Turbines,4
433,Goonabarn Farm Wind Turbines,1
467,Carland Cross Wind Farm Repowering,20
820,Land near Ventonteague,2.3


In [20]:
farm_grid_mapping["Installed Capacity (MWelec)"] = pd.to_numeric(
    farm_grid_mapping[
        "Installed Capacity (MWelec)"
    ],
    errors="coerce"
)

In [21]:
farm_grid_mapping.dtypes

Site Name                       object
Installed Capacity (MWelec)    float64
farm_lat                       float64
farm_lon                       float64
era5_lat                       float64
era5_lon                       float64
dtype: object

In [22]:
grid_capacity = (
    farm_grid_mapping
    .groupby(
        ["era5_lat", "era5_lon"]
    )
    ["Installed Capacity (MWelec)"]
    .sum()
    .reset_index()
)

In [23]:
grid_capacity["Installed Capacity (MWelec)"] = pd.to_numeric(
    grid_capacity["Installed Capacity (MWelec)"],
    errors="coerce"
)

In [24]:
grid_capacity[
    "Installed Capacity (MWelec)"
].sort_values(ascending=False).head(20)

136    1320.0
128    1218.0
246    1075.0
102    1037.7
284     950.0
130     906.2
103     857.0
49      857.0
61      714.0
106     666.0
129     659.0
29      630.0
211     630.0
92      612.0
283     588.0
186     554.7
205     544.8
237     450.0
303     449.7
227     433.9
Name: Installed Capacity (MWelec), dtype: float64

In [41]:
grid_capacity.to_parquet(
    "../data/interim/weather_processed/grid_capacity.parquet",
    index=False
)

In [25]:
total_capacity = (
    grid_capacity[
        "Installed Capacity (MWelec)"
    ].sum()
)

In [26]:
grid_capacity["weight"] = (
    grid_capacity[
        "Installed Capacity (MWelec)"
    ]
    /total_capacity
)

In [27]:
grid_capacity.head()

,era5_lat,era5_lon,Installed Capacity (MWelec),weight
0,50.00,-5.25,12.0,0.000395
1,50.25,-5.25,16.2,0.000533
2,50.25,-5.00,27.3,0.000898
3,50.50,-5.00,19.8,0.000651
4,50.50,-4.75,14.6,0.000480


In [28]:
grid_capacity.isna().sum()

era5_lat                       0
era5_lon                       0
Installed Capacity (MWelec)    0
weight                         0
dtype: int64

In [29]:
weighted_weather = None

In [30]:
for _, row in grid_capacity.iterrows():

    lat = row["era5_lat"]
    lon = row["era5_lon"]

    weight = row["weight"]

    cell_weather = weather_feature.sel(
        latitude=lat,
        longitude=lon
    )

    cell_weather = cell_weather * weight


    if weighted_weather is None:

        weighted_weather = cell_weather

    else:

        weighted_weather = (
            weighted_weather + cell_weather
        )

        

In [31]:
weighted_weather_df = (
    weighted_weather
    .to_dataframe()
    .reset_index()
)

In [32]:
weighted_weather_df.head()

,valid_time,number,expver,latitude,ws100,wd100,ws10,wd10,wind_shear,relative_humidity,air_density,blh,tcc,tp,ssrd,i10fg
0,2018-01-01 00:00:00,0,0001,60.75,12.281535,250.820054,8.888463,196.219967,1.477310,85.497575,1.221705,963.648702,0.865433,0.000137,0.0,13.781487
1,2018-01-01 01:00:00,0,0001,60.75,12.115093,248.852093,8.729048,198.381376,1.485777,84.710206,1.222602,978.021734,0.768974,0.000084,0.0,13.967654
2,2018-01-01 02:00:00,0,0001,60.75,11.805504,245.666682,8.436531,201.805059,1.493257,84.066922,1.223426,946.694984,0.637181,0.000076,0.0,13.612556
3,2018-01-01 03:00:00,0,0001,60.75,11.557783,242.461919,8.252082,205.148087,1.499402,84.011453,1.224199,902.616008,0.660511,0.000086,0.0,12.992998
4,2018-01-01 04:00:00,0,0001,60.75,11.096890,239.026854,7.947670,208.674549,1.496604,84.741920,1.224817,845.336124,0.651387,0.000078,0.0,12.466838


In [33]:
weighted_weather_df.columns.tolist()

['valid_time',
 'number',
 'expver',
 'latitude',
 'ws100',
 'wd100',
 'ws10',
 'wd10',
 'wind_shear',
 'relative_humidity',
 'air_density',
 'blh',
 'tcc',
 'tp',
 'ssrd',
 'i10fg']

In [34]:
weighted_weather_df.count()

valid_time           70128
number               70128
expver               70128
latitude             70128
ws100                70128
wd100                70128
ws10                 70128
wd10                 70128
wind_shear           70128
relative_humidity    70128
air_density          70128
blh                  70128
tcc                  70128
tp                   70128
ssrd                 70128
i10fg                70128
dtype: int64

In [75]:
weighted_weather.to_csv(
    "../data/interim/weather_processed/weighted_weather.csv",
    index=False
)

In [59]:
weighted_weather = pd.read_parquet(
    "../data/interim/weather_processed/weighted_weather.parquet"
)

In [60]:
wind_generation_hourly = pd.read_parquet(
    "../data/interim/neso_processed/wind_generation_hourly.parquet"
)

In [72]:
weighted_weather["timestamp"].min()
weighted_weather["timestamp"].max()


Timestamp('2025-12-31 23:00:00')

In [65]:

wind_generation_hourly["timestamp"].min()
wind_generation_hourly["timestamp"].max()

Timestamp('2025-12-31 23:00:00')

In [76]:
vws_dataset = weighted_weather.merge(
    wind_generation_hourly,
    on="timestamp",
    how="inner"
)

In [77]:
vws_dataset.shape

(70128, 21)

In [78]:
vws_dataset.isna().sum()

timestamp                   0
number                      0
expver                      0
latitude                    0
ws100                       0
wd100                       0
ws10                        0
wd10                        0
wind_shear                  0
relative_humidity           0
air_density                 0
blh                         0
tcc                         0
tp                          0
ssrd                        0
i10fg                       0
EMBEDDED_WIND_GENERATION    8
EMBEDDED_WIND_CAPACITY      8
CAPACITY_FACTOR             8
ND                          8
TSD                         8
dtype: int64

In [80]:
vws_dataset[
    vws_dataset["EMBEDDED_WIND_GENERATION"].isna()
][["timestamp"]]

,timestamp
2015,2018-03-25 23:00:00
10919,2019-03-31 23:00:00
19655,2020-03-29 23:00:00
28391,2021-03-28 23:00:00
37127,2022-03-27 23:00:00
45863,2023-03-26 23:00:00
54767,2024-03-31 23:00:00
63503,2025-03-30 23:00:00


In [81]:
vws_dataset = vws_dataset.dropna()

In [82]:
vws_dataset.isna().sum()

timestamp                   0
number                      0
expver                      0
latitude                    0
ws100                       0
wd100                       0
ws10                        0
wd10                        0
wind_shear                  0
relative_humidity           0
air_density                 0
blh                         0
tcc                         0
tp                          0
ssrd                        0
i10fg                       0
EMBEDDED_WIND_GENERATION    0
EMBEDDED_WIND_CAPACITY      0
CAPACITY_FACTOR             0
ND                          0
TSD                         0
dtype: int64

In [79]:
vws_dataset.head()

,timestamp,number,expver,latitude,ws100,wd100,ws10,wd10,wind_shear,relative_humidity,...,blh,tcc,tp,ssrd,i10fg,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,CAPACITY_FACTOR,ND,TSD
0,2018-01-01 00:00:00,0,0001,60.75,12.281535,250.820054,8.888463,196.219967,1.477310,85.497575,...,963.648702,0.865433,0.000137,0.0,13.781487,3034.5,5754.0,0.527372,25971.0,26854.0
1,2018-01-01 01:00:00,0,0001,60.75,12.115093,248.852093,8.729048,198.381376,1.485777,84.710206,...,978.021734,0.768974,0.000084,0.0,13.967654,2922.5,5754.0,0.507908,25766.0,27120.0
2,2018-01-01 02:00:00,0,0001,60.75,11.805504,245.666682,8.436531,201.805059,1.493257,84.066922,...,946.694984,0.637181,0.000076,0.0,13.612556,2891.0,5754.0,0.502433,24110.5,25954.0
3,2018-01-01 03:00:00,0,0001,60.75,11.557783,242.461919,8.252082,205.148087,1.499402,84.011453,...,902.616008,0.660511,0.000086,0.0,12.992998,2803.0,5754.0,0.487139,22527.0,25020.0
4,2018-01-01 04:00:00,0,0001,60.75,11.096890,239.026854,7.947670,208.674549,1.496604,84.741920,...,845.336124,0.651387,0.000078,0.0,12.466838,2638.5,5754.0,0.458551,21389.5,23867.5


In [84]:
vws_dataset.to_csv(
    "../data/processed/vws_dataset.csv",
    index=False
)